# Step 4: Join Strategy Deep Dive

## Learning Objectives
1. Understand Spark's 5 join strategies
2. Understand how each strategy works and compare execution plans
3. Criteria for choosing a join strategy
4. Skew Join optimization
5. How join order affects performance
6. Real-world join optimization patterns

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
import time

spark = SparkSession.builder \
    .appName("Step4-Join-Strategies") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/data/warehouse") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")
print(f"  → -1 = auto Broadcast disabled (for manual control)")
print(f"✅ Spark UI: http://localhost:4040")

Broadcast threshold: -1
  → -1 = auto Broadcast disabled (for manual control)
✅ Spark UI: http://localhost:4040


---
## 1. Generating Practice Data

In [2]:
random.seed(42)

# Large table: orders (1M rows)
orders = [
    (i, random.randint(1, 100000), random.randint(1, 5000),
     random.randint(1, 10), f"2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}")
    for i in range(1, 1_000_001)
]
orders_df = spark.createDataFrame(orders, ["order_id", "customer_id", "product_id", "quantity", "order_date"])

# Large table: customers (100K rows)
regions = ["Seoul", "Gyeonggi", "Busan", "Daegu", "Incheon", "Gwangju", "Daejeon", "Jeju"]
customers = [
    (i, f"customer_{i}", random.choice(regions), random.randint(20, 60))
    for i in range(1, 100_001)
]
customers_df = spark.createDataFrame(customers, ["customer_id", "name", "region", "age"])

# Small table: products (5K rows)
categories = ["Electronics", "Apparel", "Food", "Books", "Sports", "Furniture"]
products = [
    (i, f"product_{i}", random.choice(categories), random.randint(1000, 500000))
    for i in range(1, 5001)
]
products_df = spark.createDataFrame(products, ["product_id", "product_name", "category", "price"])

# Very small table: region info (8 rows)
region_info = spark.createDataFrame([
    ("Seoul",    "Capital Area",   9_700_000),
    ("Gyeonggi", "Capital Area",  13_500_000),
    ("Busan",    "Yeongnam Area",  3_400_000),
    ("Daegu",    "Yeongnam Area",  2_400_000),
    ("Incheon",  "Capital Area",   2_900_000),
    ("Gwangju",  "Honam Area",     1_500_000),
    ("Daejeon",  "Chungcheong",    1_500_000),
    ("Jeju",     "Jeju Area",        670_000)
], ["region", "area", "population"])

# Cache
orders_df.cache().count()
customers_df.cache().count()
products_df.cache().count()

print(f"Orders:    {orders_df.count():,}")
print(f"Customers: {customers_df.count():,}")
print(f"Products:  {products_df.count():,}")
print(f"Regions:   {region_info.count()}")

Orders:    1,000,000
Customers: 100,000
Products:  5,000
Regions:   8


---
## 2. Spark's 5 Join Strategies

```
┌──────────────────────────────────────────────────────────────┐
│                    Spark Join Strategy Selection             │
├──────────────────────┬───────────┬───────────────────────────┤
│ Strategy             │ Shuffle   │ Best situation            │
├──────────────────────┼───────────┼───────────────────────────┤
│ Broadcast Hash Join  │ None      │ One side is a small table │
│ Sort-Merge Join      │ Both      │ Both tables large (default)│
│ Shuffle Hash Join    │ Both      │ One side medium-sized     │
│ Broadcast Nested     │ None      │ Small table + non-equi    │
│   Loop Join          │           │   join                    │
│ Cartesian Product    │ None/Both │ No join condition (danger!)│
└──────────────────────┴───────────┴───────────────────────────┘
```

### 2.1 Broadcast Hash Join (BHJ)

Copy (broadcast) the small table to all executors, build a hash table, then join.

```
Driver
  │  send small table to each executor
  ▼
┌──────────┐  ┌──────────┐  ┌──────────┐
│Executor 0│  │Executor 1│  │Executor 2│
│          │  │          │  │          │
│ large P0 │  │ large P1 │  │ large P2 │
│  ⊕       │  │  ⊕       │  │  ⊕       │
│ small(all)│  │ small(all)│  │ small(all)│
│  → join  │  │  → join  │  │  → join  │
└──────────┘  └──────────┘  └──────────┘
    No Shuffle!
```

**Condition:** One table is smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10MB)

In [3]:
# Broadcast Hash Join: explicit broadcast() hint on the small (region_info) side
bhj_real = (
    customers_df
    .join(F.broadcast(region_info), "region")
    .select("customer_id", "name", "region", "area", "population")
)

print("=== Broadcast Hash Join Execution Plan ===")
bhj_real.explain()

start = time.time()
bhj_real.count()
bhj_time = time.time() - start
print(f"\nElapsed: {bhj_time:.3f}s")
print("\n💡 BroadcastHashJoin — No Exchange (Shuffle)! The small side is sent via BroadcastExchange.")

=== Broadcast Hash Join Execution Plan ===
== Physical Plan ==
*(2) Project [customer_id#10L, name#11, region#12, area#27, population#28L]
+- *(2) BroadcastHashJoin [region#12], [region#26], Inner, BuildRight, false
   :- *(2) Filter isnotnull(region#12)
   :  +- InMemoryTableScan [customer_id#10L, name#11, region#12], [isnotnull(region#12)]
   :        +- InMemoryRelation [customer_id#10L, name#11, region#12, age#13L], StorageLevel(disk, memory, deserialized, 1 replicas)
   :              +- *(1) Scan ExistingRDD[customer_id#10L,name#11,region#12,age#13L]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=228]
      +- *(1) Filter isnotnull(region#26)
         +- *(1) Scan ExistingRDD[region#26,area#27,population#28L]



Elapsed: 0.290s

💡 BroadcastHashJoin — No Exchange (Shuffle)! The small side is sent via BroadcastExchange.


### 2.2 Sort-Merge Join (SMJ)

Sort both tables by the join key, then merge-join them.

```
Table A                      Table B
┌───────┐  Shuffle+Sort  ┌───────┐  Shuffle+Sort
│ random │ ────────────→ │ sorted │
└───────┘               └───────┘
                              │
                         Merge Join  ← two pointers advance in order
                              │
                         ┌───────┐
                         │ result │
                         └───────┘
```

**Characteristics:** Both sides require Shuffle + Sort. Default strategy for large equi-joins.

In [4]:
# Sort-Merge Join: between large tables (Broadcast disabled)
smj = orders_df.join(customers_df, "customer_id")

print("=== Sort-Merge Join Execution Plan ===")
smj.explain()

start = time.time()
smj.count()
smj_time = time.time() - start
print(f"\nElapsed: {smj_time:.3f}s")
print("\n💡 SortMergeJoin — Exchange (Shuffle) + Sort on both sides.")

=== Sort-Merge Join Execution Plan ===
== Physical Plan ==
*(5) Project [customer_id#1L, order_id#0L, product_id#2L, quantity#3L, order_date#4, name#11, region#12, age#13L]
+- *(5) SortMergeJoin [customer_id#1L], [customer_id#10L], Inner
   :- *(2) Sort [customer_id#1L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#1L, 20), ENSURE_REQUIREMENTS, [plan_id=342]
   :     +- *(1) Filter isnotnull(customer_id#1L)
   :        +- InMemoryTableScan [order_id#0L, customer_id#1L, product_id#2L, quantity#3L, order_date#4], [isnotnull(customer_id#1L)]
   :              +- InMemoryRelation [order_id#0L, customer_id#1L, product_id#2L, quantity#3L, order_date#4], StorageLevel(disk, memory, deserialized, 1 replicas)
   :                    +- *(1) Scan ExistingRDD[order_id#0L,customer_id#1L,product_id#2L,quantity#3L,order_date#4]
   +- *(4) Sort [customer_id#10L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#10L, 20), ENSURE_REQUIREMENTS, [plan_id=3

### 2.3 Shuffle Hash Join (SHJ)

Shuffle both sides, build a hash table on the smaller side, then join. No sort needed.

**Difference from SMJ:** Uses hash table without sorting → saves sort cost, uses more memory

In [5]:
# Force Shuffle Hash Join (using hint)
shj = orders_df.hint("SHUFFLE_HASH").join(customers_df, "customer_id")

print("=== Shuffle Hash Join Execution Plan ===")
shj.explain()

start = time.time()
shj.count()
shj_time = time.time() - start
print(f"\nElapsed: {shj_time:.3f}s")
print("\n💡 ShuffledHashJoin — Exchange present but no Sort.")

=== Shuffle Hash Join Execution Plan ===
== Physical Plan ==
*(3) Project [customer_id#1L, order_id#0L, product_id#2L, quantity#3L, order_date#4, name#11, region#12, age#13L]
+- *(3) ShuffledHashJoin [customer_id#1L], [customer_id#10L], Inner, BuildLeft
   :- Exchange hashpartitioning(customer_id#1L, 20), ENSURE_REQUIREMENTS, [plan_id=499]
   :  +- *(1) Filter isnotnull(customer_id#1L)
   :     +- InMemoryTableScan [order_id#0L, customer_id#1L, product_id#2L, quantity#3L, order_date#4], [isnotnull(customer_id#1L)]
   :           +- InMemoryRelation [order_id#0L, customer_id#1L, product_id#2L, quantity#3L, order_date#4], StorageLevel(disk, memory, deserialized, 1 replicas)
   :                 +- *(1) Scan ExistingRDD[order_id#0L,customer_id#1L,product_id#2L,quantity#3L,order_date#4]
   +- Exchange hashpartitioning(customer_id#10L, 20), ENSURE_REQUIREMENTS, [plan_id=504]
      +- *(2) Filter isnotnull(customer_id#10L)
         +- InMemoryTableScan [customer_id#10L, name#11, region#12, a

### 2.4 Broadcast Nested Loop Join (BNLJ)

Broadcast the small table and compare every row combination. Used for **non-equi joins**.

**Warning:** O(N×M) complexity → very slow when both sides are large

In [6]:
# Non-equi join: find the level that matches the salary range
salary_bands = spark.createDataFrame([
    ("Junior", 50000, 80000),
    ("Mid", 80001, 110000),
    ("Senior", 110001, 150000),
], ["level", "min_salary", "max_salary"])

employees = spark.createDataFrame([
    (1, "Alice", 75000),
    (2, "Bob", 95000),
    (3, "Charlie", 120000),
    (4, "Diana", 60000),
], ["emp_id", "name", "salary"])

# Non-equi join: salary BETWEEN min_salary AND max_salary
bnlj = employees.join(
    F.broadcast(salary_bands),
    (employees.salary >= salary_bands.min_salary) & 
    (employees.salary <= salary_bands.max_salary)
)

print("=== Broadcast Nested Loop Join Execution Plan ===")
bnlj.explain()
bnlj.select("name", "salary", "level").show()

print("💡 BroadcastNestedLoopJoin — used for non-equi joins")
print("   Always broadcast() the smaller side for performance.")

=== Broadcast Nested Loop Join Execution Plan ===
== Physical Plan ==
*(2) BroadcastNestedLoopJoin BuildRight, Inner, ((salary#1902L >= min_salary#1895L) AND (salary#1902L <= max_salary#1896L))
:- *(2) Filter isnotnull(salary#1902L)
:  +- *(2) Scan ExistingRDD[emp_id#1900L,name#1901,salary#1902L]
+- BroadcastExchange IdentityBroadcastMode, [plan_id=629]
   +- *(1) Filter (isnotnull(min_salary#1895L) AND isnotnull(max_salary#1896L))
      +- *(1) Scan ExistingRDD[level#1894,min_salary#1895L,max_salary#1896L]


+-------+------+------+
|   name|salary| level|
+-------+------+------+
|  Alice| 75000|Junior|
|    Bob| 95000|   Mid|
|Charlie|120000|Senior|
|  Diana| 60000|Junior|
+-------+------+------+

💡 BroadcastNestedLoopJoin — used for non-equi joins
   Always broadcast() the smaller side for performance.


### 2.5 Cartesian Product (Cross Join)

Generates every combination of rows with no join condition. **Very dangerous!**

In [7]:
# Cross Join: demonstrated with small data only
left = spark.createDataFrame([(1,"A"), (2,"B"), (3,"C")], ["id", "val"])
right = spark.createDataFrame([("X",), ("Y",)], ["code"])

cross = left.crossJoin(right)

print("=== Cross Join ===")
cross.explain()
cross.show()

print(f"Row count: {left.count()} × {right.count()} = {cross.count()}")
print("\n⚠️ 1M × 100K = 100B rows → never do this!")
print("   Cases requiring Cross Join are extremely rare.")

=== Cross Join ===
== Physical Plan ==
CartesianProduct
:- *(1) Scan ExistingRDD[id#1934L,val#1935]
+- *(2) Scan ExistingRDD[code#1938]


+---+---+----+
| id|val|code|
+---+---+----+
|  1|  A|   X|
|  1|  A|   Y|
|  2|  B|   X|
|  2|  B|   Y|
|  3|  C|   X|
|  3|  C|   Y|
+---+---+----+

Row count: 3 × 2 = 6

⚠️ 1M × 100K = 100B rows → never do this!
   Cases requiring Cross Join are extremely rare.


---
## 3. Join Strategy Performance Comparison

In [8]:
# Run the same join with different strategies and compare
results = {}

# 1. Sort-Merge Join (default)
start = time.time()
orders_df.join(products_df, "product_id").count()
results["SortMergeJoin"] = time.time() - start

# 2. Broadcast Hash Join
start = time.time()
orders_df.join(F.broadcast(products_df), "product_id").count()
results["BroadcastHashJoin"] = time.time() - start

# 3. Shuffle Hash Join
start = time.time()
orders_df.hint("SHUFFLE_HASH").join(products_df, "product_id").count()
results["ShuffleHashJoin"] = time.time() - start

print("=== Join strategy performance (orders 1M × products 5K) ===")
print(f"{'Strategy':<25} {'Time':>8} {'Ratio':>8}")
print("-" * 45)
baseline = results["SortMergeJoin"]
for strategy, elapsed in sorted(results.items(), key=lambda x: x[1]):
    ratio = elapsed / baseline
    bar = "█" * int(ratio * 20)
    print(f"{strategy:<25} {elapsed:>7.3f}s {ratio:>7.2f}x  {bar}")

=== Join strategy performance (orders 1M × products 5K) ===
Strategy                      Time    Ratio
---------------------------------------------
BroadcastHashJoin           0.159s    0.61x  ████████████
ShuffleHashJoin             0.195s    0.75x  ██████████████
SortMergeJoin               0.261s    1.00x  ████████████████████


In [9]:
# Large table × large table comparison
results_big = {}

# Sort-Merge Join
start = time.time()
orders_df.join(customers_df, "customer_id").count()
results_big["SortMergeJoin"] = time.time() - start

# Broadcast Hash Join (broadcast customers_df — somewhat large)
start = time.time()
orders_df.join(F.broadcast(customers_df), "customer_id").count()
results_big["BroadcastHashJoin"] = time.time() - start

# Shuffle Hash Join
start = time.time()
orders_df.hint("SHUFFLE_HASH").join(customers_df, "customer_id").count()
results_big["ShuffleHashJoin"] = time.time() - start

print("=== Join strategy performance (orders 1M × customers 100K) ===")
print(f"{'Strategy':<25} {'Time':>8} {'Ratio':>8}")
print("-" * 45)
baseline = results_big["SortMergeJoin"]
for strategy, elapsed in sorted(results_big.items(), key=lambda x: x[1]):
    ratio = elapsed / baseline
    bar = "█" * int(ratio * 20)
    print(f"{strategy:<25} {elapsed:>7.3f}s {ratio:>7.2f}x  {bar}")

print("\n💡 The benefit of Broadcast diminishes as the table grows larger.")
print("   If the broadcast target exceeds executor memory, OOM risk!")

=== Join strategy performance (orders 1M × customers 100K) ===
Strategy                      Time    Ratio
---------------------------------------------
BroadcastHashJoin           0.144s    0.60x  ███████████
ShuffleHashJoin             0.174s    0.72x  ██████████████
SortMergeJoin               0.241s    1.00x  ████████████████████

💡 The benefit of Broadcast diminishes as the table grows larger.
   If the broadcast target exceeds executor memory, OOM risk!


---
## 4. Join Strategy Selection Criteria

```
                  Equi join?
                 /          \
               Yes           No
              /                \
      One side small?    One side small?
       /         \           /         \
     Yes         No        Yes         No
      │           │         │           │
  Broadcast    Both       Broadcast   Cartesian
  Hash Join    tables     Nested      Product
               large      Loop Join   (danger!)
               /     \
         Sort needed? Enough memory?
          /    \       /     \
        Yes    No    Yes     No
         │      │     │      │
      Sort-   Sort-  Shuffle Sort-
      Merge   Merge  Hash    Merge
      Join    Join   Join    Join
```

In [10]:
# Comprehensive summary of Join hints
print("""
=== Spark Join Hints ===

1. BROADCAST / BROADCASTJOIN / MAPJOIN
   df.hint("BROADCAST")  or  F.broadcast(df)
   → Forces BroadcastHashJoin

2. MERGE / SHUFFLE_MERGE / MERGEJOIN
   df.hint("MERGE")
   → Forces SortMergeJoin

3. SHUFFLE_HASH
   df.hint("SHUFFLE_HASH")
   → Forces ShuffledHashJoin

4. SHUFFLE_REPLICATE_NL
   df.hint("SHUFFLE_REPLICATE_NL")
   → Forces CartesianProduct

SQL hint syntax:
   SELECT /*+ BROADCAST(small_table) */ ...
   SELECT /*+ MERGE(t1, t2) */ ...
""")


=== Spark Join Hints ===

1. BROADCAST / BROADCASTJOIN / MAPJOIN
   df.hint("BROADCAST")  or  F.broadcast(df)
   → Forces BroadcastHashJoin

2. MERGE / SHUFFLE_MERGE / MERGEJOIN
   df.hint("MERGE")
   → Forces SortMergeJoin

3. SHUFFLE_HASH
   df.hint("SHUFFLE_HASH")
   → Forces ShuffledHashJoin

4. SHUFFLE_REPLICATE_NL
   df.hint("SHUFFLE_REPLICATE_NL")
   → Forces CartesianProduct

SQL hint syntax:
   SELECT /*+ BROADCAST(small_table) */ ...
   SELECT /*+ MERGE(t1, t2) */ ...



In [11]:
# SQL hint examples
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
customers_df.createOrReplaceTempView("customers")

# Using hints in SQL
bhj_sql = spark.sql("""
    SELECT /*+ BROADCAST(p) */
        o.order_id, p.product_name, p.price
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
""")

print("=== SQL Broadcast Hint ===")
bhj_sql.explain()

smj_sql = spark.sql("""
    SELECT /*+ MERGE(o, c) */
        o.order_id, c.name
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
""")

print("\n=== SQL Merge Hint ===")
smj_sql.explain()

=== SQL Broadcast Hint ===
== Physical Plan ==
*(2) Project [order_id#0L, product_name#19, price#21L]
+- *(2) BroadcastHashJoin [product_id#2L], [product_id#18L], Inner, BuildRight, false
   :- *(2) Filter isnotnull(product_id#2L)
   :  +- InMemoryTableScan [order_id#0L, product_id#2L], [isnotnull(product_id#2L)]
   :        +- InMemoryRelation [order_id#0L, customer_id#1L, product_id#2L, quantity#3L, order_date#4], StorageLevel(disk, memory, deserialized, 1 replicas)
   :              +- *(1) Scan ExistingRDD[order_id#0L,customer_id#1L,product_id#2L,quantity#3L,order_date#4]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=1363]
      +- *(1) Filter isnotnull(product_id#18L)
         +- InMemoryTableScan [product_id#18L, product_name#19, price#21L], [isnotnull(product_id#18L)]
               +- InMemoryRelation [product_id#18L, product_name#19, category#20, price#21L], StorageLevel(disk, memory, deserialized, 1 replicas)
              

---
## 5. Join Types (Inner, Left, Right, Full, Semi, Anti)

In [12]:
# Small demo tables
left_df = spark.createDataFrame([
    (1, "Alice"), (2, "Bob"), (3, "Charlie"), (4, "Diana")
], ["id", "name"])

right_df = spark.createDataFrame([
    (1, "Engineering"), (2, "Marketing"), (5, "Sales")
], ["id", "department"])

join_types = ["inner", "left", "right", "full", "left_semi", "left_anti"]

for jt in join_types:
    print(f"\n{'=' * 40}")
    print(f"  {jt.upper()} JOIN")
    print(f"{'=' * 40}")
    left_df.join(right_df, "id", jt).show()


  INNER JOIN
+---+-----+-----------+
| id| name| department|
+---+-----+-----------+
|  2|  Bob|  Marketing|
|  1|Alice|Engineering|
+---+-----+-----------+


  LEFT JOIN
+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  4|  Diana|       NULL|
|  3|Charlie|       NULL|
|  2|    Bob|  Marketing|
|  1|  Alice|Engineering|
+---+-------+-----------+


  RIGHT JOIN
+---+-----+-----------+
| id| name| department|
+---+-----+-----------+
|  2|  Bob|  Marketing|
|  5| NULL|      Sales|
|  1|Alice|Engineering|
+---+-----+-----------+


  FULL JOIN
+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  4|  Diana|       NULL|
|  3|Charlie|       NULL|
|  2|    Bob|  Marketing|
|  5|   NULL|      Sales|
|  1|  Alice|Engineering|
+---+-------+-----------+


  LEFT_SEMI JOIN
+---+-----+
| id| name|
+---+-----+
|  2|  Bob|
|  1|Alice|
+---+-----+


  LEFT_ANTI JOIN
+---+-------+
| id|   name|
+---+-------+
|  4|  Diana|
|  3|Charlie|
+---+----

In [13]:
print("""
💡 Semi Join vs Anti Join — commonly used patterns in practice

LEFT SEMI JOIN:
  "Return only rows from the left table that have a match on the right"
  = equivalent to EXISTS / IN subquery
  → Extract only customers who have orders

LEFT ANTI JOIN:
  "Return only rows from the left table that have no match on the right"
  = equivalent to NOT EXISTS / NOT IN subquery
  → Extract customers who have never ordered

✅ Semi/Anti do not return right-side columns, so less data is transferred.
""")

# Practical example: customers with order history (Semi Join)
active_customers = customers_df.join(orders_df, "customer_id", "left_semi")
print(f"Customers with orders: {active_customers.count():,}")

# Practical example: customers who have never ordered (Anti Join)
inactive_customers = customers_df.join(orders_df, "customer_id", "left_anti")
print(f"Customers without orders: {inactive_customers.count():,}")


💡 Semi Join vs Anti Join — commonly used patterns in practice

LEFT SEMI JOIN:
  "Return only rows from the left table that have a match on the right"
  = equivalent to EXISTS / IN subquery
  → Extract only customers who have orders

LEFT ANTI JOIN:
  "Return only rows from the left table that have no match on the right"
  = equivalent to NOT EXISTS / NOT IN subquery
  → Extract customers who have never ordered

✅ Semi/Anti do not return right-side columns, so less data is transferred.

Customers with orders: 99,993
Customers without orders: 7


---
## 6. Skew Join Optimization

In [14]:
# Generate skewed data: 80% concentrated on customer_id = 1
random.seed(42)
skew_orders = []
for i in range(500_000):
    if random.random() < 0.8:
        cid = 1  # 80% goes to a single customer
    else:
        cid = random.randint(2, 10000)
    skew_orders.append((i, cid, random.randint(1, 5000), random.randint(1, 10)))

skew_orders_df = spark.createDataFrame(skew_orders, ["order_id", "customer_id", "product_id", "quantity"])

# Confirm skew
print("=== customer_id distribution (top 5) ===")
skew_orders_df.groupBy("customer_id").count() \
    .orderBy(F.col("count").desc()) \
    .show(5)

print("⚠️ 400K rows concentrated on customer_id=1 → one partition overloaded during join")

=== customer_id distribution (top 5) ===
+-----------+------+
|customer_id| count|
+-----------+------+
|          1|400113|
|       4305|    24|
|       6172|    24|
|       6101|    23|
|       3198|    23|
+-----------+------+
only showing top 5 rows

⚠️ 400K rows concentrated on customer_id=1 → one partition overloaded during join


In [15]:
# Observe Skew Join performance problem
small_customers = customers_df.limit(10000).cache()
small_customers.count()

start = time.time()
skew_orders_df.join(small_customers, "customer_id").count()
skew_join_time = time.time() - start
print(f"Skew Join time: {skew_join_time:.3f}s")

Skew Join time: 0.273s


In [16]:
# Fix 1: Salting — split the hot key into num_salts sub-keys so it no longer floods one partition
num_salts = 20

# --- Deterministic evidence: size of the hottest reduce partition (the straggler task) ---
def hottest_partition(df, key, n=20):
    counts = [r["c"] for r in (df.repartition(n, key)
              .withColumn("p", F.spark_partition_id())
              .groupBy("p").agg(F.count("*").alias("c")).collect())]
    return max(counts)

before_hot = hottest_partition(skew_orders_df, "customer_id")

salted_orders = skew_orders_df.withColumn(
    "salt", (F.rand() * num_salts).cast("int")
).withColumn(
    "salted_key", F.concat(F.col("customer_id").cast("string"), F.lit("_"), F.col("salt").cast("string"))
)
after_hot = hottest_partition(salted_orders, "salted_key")
print(f"Hottest partition BEFORE salting (by customer_id): {before_hot:,} rows")
print(f"Hottest partition AFTER  salting (by salted_key) : {after_hot:,} rows  -> ~{before_hot/after_hot:.0f}x smaller\n")

# Replicate the SMALL side by the number of salts (explode), then join on the salted key
start = time.time()
salted_customers = small_customers.withColumn(
    "salt", F.explode(F.array([F.lit(i) for i in range(num_salts)]))
).withColumn(
    "salted_key", F.concat(F.col("customer_id").cast("string"), F.lit("_"), F.col("salt").cast("string"))
)
salted_result = salted_orders.join(salted_customers, "salted_key").drop("salted_key", "salt")
salted_result.count()
salt_time = time.time() - start

print(f"Regular Join: {skew_join_time:.3f}s")
print(f"Salted Join:  {salt_time:.3f}s")
print("""
⚠️  On this toy cluster the salted join is usually SLOWER, not faster — the skewed partition
   here is only ~9MB, so its straggler costs less than the salting overhead (20x replication of
   the small side + a wider shuffle). The hottest-partition numbers above are the real signal:
   salting cuts the biggest reduce task several-fold. At real scale, where the hot partition is
   GBs and the straggler runs for minutes, that reduction is the whole game.

   How it works:
     orders : customer_id=1           -> 400K rows in ONE partition
              customer_id=1_0..1_19   -> ~20K rows each across 20 partitions
   The customers side must be replicated x20 (one copy per salt) so every salted order still
   finds its match — so you always salt + replicate the SMALLER side.
""")

Hottest partition BEFORE salting (by customer_id): 405,627 rows
Hottest partition AFTER  salting (by salted_key) : 65,426 rows  -> ~6x smaller

Regular Join: 0.273s
Salted Join:  0.500s

⚠️  On this toy cluster the salted join is usually SLOWER, not faster — the skewed partition
   here is only ~9MB, so its straggler costs less than the salting overhead (20x replication of
   the small side + a wider shuffle). The hottest-partition numbers above are the real signal:
   salting cuts the biggest reduce task several-fold. At real scale, where the hot partition is
   GBs and the straggler runs for minutes, that reduction is the whole game.

   How it works:
     orders : customer_id=1           -> 400K rows in ONE partition
              customer_id=1_0..1_19   -> ~20K rows each across 20 partitions
   The customers side must be replicated x20 (one copy per salt) so every salted order still
   finds its match — so you always salt + replicate the SMALLER side.



In [17]:
# Fix 2: AQE Skew Join — splits skewed partitions automatically AT RUNTIME (no code change)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64m")

start = time.time()
skew_orders_df.join(small_customers, "customer_id").count()
aqe_time = time.time() - start

# Why does AQE change nothing here? Estimate the hot partition's size vs the split threshold.
hot_rows = skew_orders_df.filter("customer_id = 1").count()
approx_mb = hot_rows * 28 / 1024 / 1024   # ~28 bytes/row (4 longs), rough estimate

print(f"Regular Join: {skew_join_time:.3f}s")
print(f"Salted Join:  {salt_time:.3f}s")
print(f"AQE Join:     {aqe_time:.3f}s   (≈ regular — AQE did NOT split anything here)")
print(f"\nhot partition ~ {hot_rows:,} rows ≈ {approx_mb:.0f} MB   <   threshold 64 MB")
print("""
💡 AQE Skew Join is THRESHOLD-GATED — and that is the real lesson:
   a partition is split only if it is BOTH
     - >= skewedPartitionFactor (5) x the median partition size, AND
     - >= skewedPartitionThresholdInBytes (64MB).
   Our hot partition is only ~9MB, so AQE correctly leaves it alone — splitting a tiny partition
   is not worth the overhead. (Lower the threshold below ~9MB and it would start splitting.)

   At real scale a hot partition is multiple GB; AQE then splits it into sub-partitions at
   runtime with no code change — which is why AQE skewJoin is preferred over manual salting
   whenever you can enable it.
""")
spark.conf.set("spark.sql.adaptive.enabled", "false")

Regular Join: 0.273s
Salted Join:  0.500s
AQE Join:     0.249s   (≈ regular — AQE did NOT split anything here)

hot partition ~ 400,113 rows ≈ 11 MB   <   threshold 64 MB

💡 AQE Skew Join is THRESHOLD-GATED — and that is the real lesson:
   a partition is split only if it is BOTH
     - >= skewedPartitionFactor (5) x the median partition size, AND
     - >= skewedPartitionThresholdInBytes (64MB).
   Our hot partition is only ~9MB, so AQE correctly leaves it alone — splitting a tiny partition
   is not worth the overhead. (Lower the threshold below ~9MB and it would start splitting.)

   At real scale a hot partition is multiple GB; AQE then splits it into sub-partitions at
   runtime with no code change — which is why AQE skewJoin is preferred over manual salting
   whenever you can enable it.



---
## 7. Join Order Optimization

In [18]:
# 3-table join: intermediate result size varies by order

# Order 1: (large × large) × small
start = time.time()
r1 = orders_df.join(customers_df, "customer_id").join(F.broadcast(products_df), "product_id")
r1.count()
order1_time = time.time() - start

# Order 2: (large × small) × large
start = time.time()
r2 = orders_df.join(F.broadcast(products_df), "product_id").join(customers_df, "customer_id")
r2.count()
order2_time = time.time() - start

# Order 3: filter first, then join
start = time.time()
r3 = (
    orders_df.filter(F.col("quantity") >= 5)  # reduce data volume first
    .join(F.broadcast(products_df), "product_id")
    .join(customers_df, "customer_id")
)
r3.count()
order3_time = time.time() - start

print("=== Performance by Join Order ===")
print(f"(orders × customers) × products:        {order1_time:.3f}s")
print(f"(orders × products) × customers:        {order2_time:.3f}s")
print(f"filter → (orders × products) × customers: {order3_time:.3f}s")

print(f"""
💡 Join order optimization principles:
   1. Join smaller tables first (minimize intermediate result size)
   2. Apply filters before joining (Predicate Pushdown)
   3. Do broadcastable joins first
   4. Do high-selectivity joins first
   
   Catalyst auto-optimizes to some extent,
   but complex queries may still need manual order tuning.
""")

=== Performance by Join Order ===
(orders × customers) × products:        0.317s
(orders × products) × customers:        0.279s
filter → (orders × products) × customers: 0.220s

💡 Join order optimization principles:
   1. Join smaller tables first (minimize intermediate result size)
   2. Apply filters before joining (Predicate Pushdown)
   3. Do broadcastable joins first
   4. Do high-selectivity joins first
   
   Catalyst auto-optimizes to some extent,
   but complex queries may still need manual order tuning.



---
## 8. Real-World Optimization Patterns

In [19]:
# Pattern 1: deduplicate before joining (reduce join input size)

# ❌ Join all orders and customers, then distinct
start = time.time()
bad = orders_df.join(customers_df, "customer_id") \
    .select("customer_id", "name", "region") \
    .distinct() \
    .count()
bad_time = time.time() - start

# ✅ Deduplicate customer_id first, then join
start = time.time()
good = orders_df.select("customer_id").distinct() \
    .join(customers_df, "customer_id") \
    .select("customer_id", "name", "region") \
    .count()
good_time = time.time() - start

print("Pattern 1: Deduplicate before join")
print(f"  ❌ join → distinct: {bad_time:.3f}s")
print(f"  ✅ distinct → join: {good_time:.3f}s")

Pattern 1: Deduplicate before join
  ❌ join → distinct: 0.431s
  ✅ distinct → join: 0.282s


In [20]:
# Pattern 2: aggregate before joining (reduce join input size)

# ❌ Join then aggregate
start = time.time()
bad2 = orders_df.join(customers_df, "customer_id") \
    .groupBy("region") \
    .agg(F.sum("quantity").alias("total_qty")) \
    .collect()
bad2_time = time.time() - start

# ✅ Aggregate first, then join
start = time.time()
order_summary = orders_df.groupBy("customer_id") \
    .agg(F.sum("quantity").alias("total_qty"))

good2 = order_summary.join(customers_df, "customer_id") \
    .groupBy("region") \
    .agg(F.sum("total_qty").alias("total_qty")) \
    .collect()
good2_time = time.time() - start

print("Pattern 2: Aggregate before join")
print(f"  ❌ join → aggregate: {bad2_time:.3f}s")
print(f"  ✅ aggregate → join: {good2_time:.3f}s")

Pattern 2: Aggregate before join
  ❌ join → aggregate: 0.452s
  ✅ aggregate → join: 0.389s


In [21]:
# Pattern 3: Map-side Join (using broadcast variable)
# For very small lookup data, broadcast variable is more efficient

# Regular join
region_lookup_df = region_info.select("region", "area")

start = time.time()
customers_df.join(F.broadcast(region_lookup_df), "region").count()
join_time = time.time() - start

# Broadcast variable + map
region_map = dict(region_info.select("region", "area").collect())
region_bc = sc.broadcast(region_map)

# Mapping via UDF (suitable for small lookups)
from pyspark.sql.functions import udf

@udf("string")
def lookup_area(region):
    return region_bc.value.get(region)

start = time.time()
customers_df.withColumn("area", lookup_area(F.col("region"))).count()
bc_time = time.time() - start

# Better approach: create_map (built-in function)
mapping_expr = F.create_map(
    *[item for pair in region_map.items() for item in (F.lit(pair[0]), F.lit(pair[1]))]
)

start = time.time()
customers_df.withColumn("area", mapping_expr[F.col("region")]).count()
map_time = time.time() - start

print("Pattern 3: Small lookup mapping")
print(f"  Broadcast Join:    {join_time:.3f}s")
print(f"  Broadcast + UDF:   {bc_time:.3f}s")
print(f"  create_map (best): {map_time:.3f}s")
print("\n💡 For <10 lookup entries, create_map is the fastest.")
print("   Using built-in functions instead of UDF enables Catalyst optimization.")

region_bc.unpersist()

Pattern 3: Small lookup mapping
  Broadcast Join:    0.115s
  Broadcast + UDF:   0.044s
  create_map (best): 0.029s

💡 For <10 lookup entries, create_map is the fastest.
   Using built-in functions instead of UDF enables Catalyst optimization.


---
## 📝 Key Summary

| Strategy | Shuffle | Sort | Condition | When to use |
|------|---------|------|------|----------|
| **Broadcast Hash** | None | None | One side < 10MB | Small dimension tables |
| **Sort-Merge** | Both | Both | Equi join | Large × large (default) |
| **Shuffle Hash** | Both | None | Equi join | One side medium-sized |
| **Broadcast NL** | None | None | Non-equi join | Range joins, etc. |
| **Cartesian** | Possible | None | No condition | Rarely used |

### Optimization Checklist
1. ✅ Always use `broadcast()` for small tables
2. ✅ Reduce data with filters/aggregations before joining
3. ✅ Skewed data: use Salting or AQE
4. ✅ Use Semi/Anti Join (avoids transferring unnecessary columns)
5. ✅ Use `create_map` for small lookups
6. ✅ Join order: smaller tables and filters first
7. ✅ Make a habit of checking strategies with `explain()`

### Next Step (Step 5)
- Memory management & tuning
- Executor/Driver memory structure
- GC tuning, understanding Spill
- Key configuration parameters

In [22]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
